# RoadSoS NLU: Colab training notebook

This notebook runs a quick sanity training and evaluation of the EmergencyNLUModel (XLM-RoBERTa + CRF) from the repository.

Instructions:
1. Open this notebook in Google Colab (File -> Upload notebook) or upload this file into Colab.
2. Select `Runtime -> Change runtime type` and choose `GPU`. Save.
3. Provide the repository code to the notebook by either: clone from a public GitHub URL (cell below), or upload a zip of the repository and unzip it into `/content/roadsos`.
4. Run cells sequentially. The notebook will install dependencies, run a 1-epoch sanity train, then run evaluation and save reports.

If your repo is private, upload a zip or mount Google Drive containing the repository.

In [ ]:
#@title Step 1 — Repository source (choose ONE)
# Option A: clone from GitHub (set REPO_URL variable and run)
REPO_URL = ''  # @param {type: 'string'}
if REPO_URL:
    !git clone {REPO_URL} repo
    !ls -la repo

# Option B: upload zip manually if REPO_URL is empty. Use the file upload UI in Colab.
from google.colab import files
import os
if not REPO_URL:
    print('No REPO_URL set — if you have a zip of the repo, upload it now (choose file and run).')
    # The user can upload a zip and we will extract it below after upload.
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.zip'):
            !unzip -q {name} -d repo || true
            print('Extracted', name)

# The notebook expects the code under /content/repo or /content/roadsos; move if necessary.
if os.path.exists('repo') and not os.path.exists('roadsos'):
    !mv repo roadsos || true

if os.path.exists('roadsos'):
    print('Repository ready at /content/roadsos')
else:
    print('Please ensure repository is available in /content/roadsos (clone or upload).')

In [ ]:
#@title Step 2 — Install dependencies
import sys
# Use CPU/GPU appropriate torch install; this tries to install a compatible torch wheel.
!pip install -q transformers==4.40.0 torchcrf==1.1.0 seqeval==1.2.2 scikit-learn numpy
# If GPU available, install a CUDA-enabled torch via recommended wheels (Colab often has cuda 11.x/12.x).
import torch
print('PyTorch version after install:', torch.__version__)

In [ ]:
#@title Step 3 — Quick sanity training (1 epoch)
import os
os.environ['EPOCHS'] = os.environ.get('EPOCHS', '1')
os.environ['PYTHONUNBUFFERED'] = '1'
# Move into repository and run training script
%cd /content/roadsos/backend
!python -u app/nlu/train.py 2>&1 | sed -u 's/^/TRAIN: /' || true

In [ ]:
#@title Step 4 — Run evaluation and list results
%cd /content/roadsos/backend
!python -u app/nlu/evaluate.py 2>&1 | sed -u 's/^/EVAL: /' || true
!ls -la app/nlu/checkpoints || true
!ls -la app/nlu/checkpoints/results || true

In [ ]:
#@title Step 5 — Download artifacts (best_model.pt, evaluation_report.json, evaluation_report.md)
from google.colab import files
artifacts = [
    'app/nlu/checkpoints/best_model.pt',
    'app/nlu/checkpoints/results/evaluation_report.json',
    'app/nlu/checkpoints/results/evaluation_report.md',
]
for a in artifacts:
    if os.path.exists(a):
        files.download(a)
    else:
        print('Not found:', a)

## Notes and next steps
- To run a full training, set the environment variable `EPOCHS` to desired value or edit `CONFIG['epochs']` in `app/nlu/train.py`.
- For production-level training prefer a GPU with at least 12GB VRAM (e.g., aT4, P100, or better).
- If you plan to run longer experiments, consider mounting Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and saving checkpoints there.